# Project: Data Science Pipeline - Fashion Forward Forecasting

##### Data Scientist Nanodegree - Master of Science in Artificial Intelligence

##### Udacity - August 27, 2026

##### Alaa Alaboud - Dhahran - KSA

<BR>

<BR>

## Project Overview

"StyleSense", a rapidly growing online women's clothing retailer,  is known for its trendy and affordable fashion, and its customer base has exploded in recent months. 
This influx of new customers is fantastic for business, but it has created a challenge: a backlog of product reviews with missing data. Customers are still leaving valuable feedback in the text of their reviews, but they aren't always indicating whether they recommend the product.



## The target 

- **Recommended IND**: Binary variable stating where the customer recommends the product where `1` is recommended, `0` is not recommended.

## The features

##### The features can be summarized as the following:

- **Clothing ID**: Integer Categorical variable that refers to the specific piece being reviewed.
- **Age**: Positive Integer variable of the reviewers age.
- **Title**: String variable for the title of the review.
- **Review Text**: String variable for the review body.
- **Positive Feedback Count**: Positive Integer documenting the number of other customers who found this review positive.
- **Division Name**: Categorical name of the product high level division.
- **Department Name**: Categorical name of the product department name.
- **Class Name**: Categorical name of the product class name.


## Data Loading & Library Imports

This section imports the required libraries, prepares the spaCy language model, loads the review dataset, and performs an initial inspection of the data.

In [ ]:
# Import the libraries and tools needed for data processing and model building

import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import ConfusionMatrixDisplay


##### Run this command once if the spaCy English model is not installed !

In [ ]:
# !python -m spacy download en_core_web_sm

In [ ]:
# The relative path matches the supplied Udacity starter structure.
df = pd.read_csv('data/reviews.csv')

df.info()
df.head()

## Preparing features (`X`) & target (`y`)

This section separates the input features (X) from the target variable (y), then splits the data into training and test sets for model development and evaluation.

In [ ]:
data = df

# separate features from labels
X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

print('Labels:', y.unique())
print('Features:')
display(X.head())

In [ ]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    shuffle=True,
    random_state=27,
)

## Data Exploration

This section explores the training data, target distribution, and feature characteristics before building the preprocessing pipeline.

In [ ]:
# Display summary statistics for the numerical training features

X_train.describe()

In [ ]:
# Print and check the distribution of the target classes in the training data

print('\nTraining target counts:')
print(y_train.value_counts())

print('\nTraining target percentages:')
print((y_train.value_counts(normalize=True) * 100).round(2))

# Check for duplicate rows in the training features
print('Duplicate feature rows:', X_train.duplicated().sum())


### Insight 1:

The target is imbalanced: approximately **81.55%** of observations are class `1`, while **18.45%** are class `0`. Accuracy alone could therefore hide weak performance on customers who do not recommend a product.

In [ ]:
# Explore categorical features

categorical_columns = [
    'Clothing ID',
    'Division Name',
    'Department Name',
    'Class Name'
]

for column in categorical_columns:
    print(f"\n{column}")
    print(f"Number of categories: {X_train[column].nunique()}")
    print(X_train[column].value_counts())

In [ ]:
# Explore numerical features
numerical_columns = [
    'Age',
    'Positive Feedback Count'
]

for column in numerical_columns:
    print(f"\n{column}")
    print(X_train[column].describe())

## Building the Preprocessing Pipeline

This section prepares numerical, categorical, and text features using separate preprocessing steps that are later combined into a single pipeline.

In [ ]:
# Split features into numerical, categorical, and text features

num_features = (X
    .select_dtypes(exclude=['object']).columns
    .drop('Clothing ID'))

print('Numerical features:', num_features)

cat_features = (X[[
    'Clothing ID',
    'Division Name',
    'Department Name',
    'Class Name',
]].columns
               )
print('Categorical features:', cat_features)

text_features = (X[[
    'Title',
    'Review Text',
]].columns
                )
print ('Text features:', text_features)

In [ ]:
# Define pipeline for numerical features

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler())
])
num_pipeline

In [ ]:
# Define the preprocessing pipeline for categorical features
cat_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='most_frequent'),
    ),
    (
        'cat_encoder',
        OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore',
        ),
    ),
])

cat_pipeline

## Preparing spaCy for Text Processing

This section prepares the spaCy English language model and demonstrates tokenization and lemmatization before integrating NLP into the pipeline.

In [ ]:
# Load the pre-trained English spaCy language model
nlp = spacy.load('en_core_web_sm')

In [ ]:
# Get the tokens using spaCy

review_text = X_train['Review Text'].iloc[0]
title = X_train['Title'].iloc[0]

review_tokens = nlp.tokenizer(review_text)
title_tokens = nlp.tokenizer(title)

In [ ]:
# Display the tokens generated from the sample review
for token in review_tokens:
    print(token.text)

In [ ]:
# Display the tokens generated from the sample title
for token in title_tokens:
    print(token.text)

In [ ]:
diffs_review: list[tuple[str, str]] = []
# Use spaCy to compare the tokens before and after lemmatization

doc = nlp(review_text)

for token in doc:
    if token.text != token.lemma_:
        diffs_review.append((token.text, token.lemma_))

diffs_review

In [ ]:
diffs_title: list[tuple[str, str]] = []
# Use spaCy to compare the tokens before and after lemmatization

doc = nlp(title)

for token in doc:
    if token.text != token.lemma_:
        diffs_title.append((token.text, token.lemma_))

diffs_title

In [ ]:
# Define a custom transformer to count adjectives using spaCy POS tags
class AdjectiveCounter(BaseEstimator, TransformerMixin):
    def __init__(self, nlp):
        self.nlp = nlp

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        adjective_counts = []

        for doc in self.nlp.pipe(X):
            count = sum(1 for token in doc if token.pos_ == 'ADJ')
            adjective_counts.append([count])

        return np.array(adjective_counts)

## Text Preprocessing and Feature Engineering

This section preprocesses numerical, categorical, and text features, applies spaCy-based text lemmatization and TF-IDF vectorization, and combines all preprocessing steps into a unified ColumnTransformer for model training.

In [ ]:
# Define a custom spaCy transformer for text lemmatization and cleaning

class SpacyLemmatizer(BaseEstimator, TransformerMixin):
    def __init__(self, nlp):
        self.nlp = nlp

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        lemmatized_text = []

        for doc in self.nlp.pipe(X):
            lemmas = [
                token.lemma_ 
                for token in doc 
                if not token.is_stop
                and not token.is_punct
                and not token.is_space
            ]
            lemmatized_text.append(' '.join(lemmas))

        return lemmatized_text

In [ ]:
# Build the text pipeline: lemmatize text, then convert it to TF-IDF features

text_pipeline = Pipeline([
    (
        'lemmatizer',
        SpacyLemmatizer(nlp=nlp)
    ),
    (
        'tfidf',
        TfidfVectorizer()
    ),
])

text_pipeline

In [ ]:
# Combine preprocessing pipelines for numerical, categorical, and text features

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features),
    ('title', text_pipeline, 'Title'),
    ('review', text_pipeline, 'Review Text'),
    ('review_adj_count', AdjectiveCounter(nlp), 'Review Text')
])

preprocessor

## Model `1` Training Pipeline

This section builds and evaluates the first machine learning pipeline using Logistic Regression.

### Baseline Logistic Regression

This section trains and evaluates the baseline Logistic Regression model using the complete preprocessing pipeline.

In [ ]:
# Build the baseline classification pipeline using the shared preprocessor

logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=27
    ))
])

In [ ]:
# Train the baseline Logistic Regression pipeline on the training data

logistic_pipeline.fit(X_train, y_train)

In [ ]:
# Generate predictions on the test data using the baseline Logistic Regression model

y_pred_logistic = logistic_pipeline.predict(X_test)

In [ ]:
# Evaluate the baseline model using classification metrics and the confusion matrix

print("Accuracy:", accuracy_score(y_test, y_pred_logistic))
print("Precision:", precision_score(y_test, y_pred_logistic))
print("Recall:", recall_score(y_test, y_pred_logistic))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

### Insight 2:

The classifiers use balanced class weights, Logistic Regression tuning uses macro F1, and the final evaluation reports per-class precision, recall, F1-score, and a confusion matrix.

## Fine-Tuning Logistic Regression

This section uses RandomizedSearchCV to search for improved Logistic Regression hyperparameters using macro F1-score.

In [ ]:
# Define the hyperparameter search space and configure RandomizedSearchCV

param_distributions_logistic = {
    'classifier__C': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    'classifier__class_weight': ['balanced'],
    'classifier__solver': ['liblinear', 'lbfgs']
}

logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=param_distributions_logistic,
    n_iter=14,
    scoring='f1_macro',
    cv=3,
    random_state=27,
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Run the hyperparameter search on the training data

logistic_search.fit(X_train, y_train)

In [ ]:
# Display the best hyperparameters and cross-validation score

print("Best parameters:", logistic_search.best_params_)
print("Best CV score:", logistic_search.best_score_)

In [ ]:
# Select the best tuned model and generate predictions on the test data

best_logistic = logistic_search.best_estimator_
y_pred_logistic_final = best_logistic.predict(X_test)

In [ ]:
# Evaluate the tuned Logistic Regression model on the test data

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred_logistic_final)
)

print(
    "Precision:",
    precision_score(y_test, y_pred_logistic_final)
)

print(
    "Recall:",
    recall_score(y_test, y_pred_logistic_final)
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic_final))

### Insight 3:

The tuned Logistic Regression achieved a macro F1-score of **0.82** on the test set. Performance remained strong for the majority class while maintaining substantially higher recall (**0.86**) than precision (**0.61**) for the minority class, showing that the model captures most non-recommendations despite some false positives.

## Model `2` Training Pipeline

This section builds and evaluates the second machine learning pipeline using Random Forest as an alternative classifier.

## Random Forest Experiment

This section evaluates Random Forest as an alternative classification model using the same preprocessing pipeline.

In [ ]:
# Build a Random Forest pipeline using the same shared preprocessor

random_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    (
        'classifier',
        RandomForestClassifier(
            class_weight='balanced',
            random_state=27
        )
    )
])

random_forest_pipeline

In [ ]:
# Train the baseline Random Forest pipeline on the training data

random_forest_pipeline.fit(X_train, y_train)

In [ ]:
# Generate predictions on the test data using the baseline Random Forest model

y_pred_rf = random_forest_pipeline.predict(X_test)

In [ ]:
# Evaluate the baseline Random Forest model on the test data

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

### Insight 4:

The baseline Random Forest achieved approximately **88.40%** accuracy and a macro F1-score of **0.79**. While it performed strongly on class `1` with an F1-score of **0.93**, its performance on class `0` was weaker, with an F1-score of **0.65**. This indicates that the model is better at identifying customers who recommend a product than those who do not.

## Fine-Tuning Random Forest

This section uses RandomizedSearchCV to search for improved Random Forest hyperparameters using macro F1-score.

In [ ]:
# Define the hyperparameter search space and configure RandomizedSearchCV

param_distributions_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__class_weight': ['balanced']
}

random_forest_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=param_distributions_rf,
    n_iter=10,
    scoring='f1_macro',
    cv=3,
    random_state=27,
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Run the hyperparameter search on the training data
random_forest_search.fit(X_train, y_train)

In [ ]:
# Display the best hyperparameters and cross-validation score

print("Best parameters:", random_forest_search.best_params_)
print("Best CV score:", random_forest_search.best_score_)

In [ ]:
# Select the best tuned Random Forest model and generate predictions

best_random_forest = random_forest_search.best_estimator_
y_pred_rf_final = best_random_forest.predict(X_test)

In [ ]:
# Evaluate the tuned Random Forest model on the test data

print("Accuracy:", accuracy_score(y_test, y_pred_rf_final))
print("Precision:", precision_score(y_test, y_pred_rf_final))
print("Recall:", recall_score(y_test, y_pred_rf_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_final))

### Insight 5:

Fine-tuning improved the Random Forest model's performance on the minority class. The recall for class `0` increased from **0.61** to **0.75**, while its F1-score improved from **0.65** to **0.69**. Although overall accuracy decreased slightly from **88.40%** to **87.86%**, the tuned model achieved a better balance between the two classes, with a macro F1-score of **0.81**.

## Model Comparison

This section compares the trained models using accuracy, macro F1-score, and weighted F1-score to select the final model.

| Model | Accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|
| Baseline Logistic Regression | 0.881 | 0.821 | 0.888 |
| Tuned Logistic Regression | 0.879 | 0.819 | 0.886 |
| Baseline Random Forest | 0.884 | 0.790 | 0.881 |
| Tuned Random Forest | 0.879 | 0.806 | 0.883 |

### Insight 6:

Although the baseline Random Forest achieved the highest overall accuracy, the baseline Logistic Regression achieved the highest macro F1-score **(0.821)** and showed more balanced performance across both classes. Since the target is imbalanced, the baseline Logistic Regression was selected as the final model.

## Final Model Evaluation

This section evaluates the selected final model on the test data using classification metrics and a confusion matrix.

### Baseline Logistic Regression

The **Baseline Logistic Regression** model was selected as the final model because it achieved the highest Macro F1-score and provided the best balance between the two target classes.

In [ ]:
# Use the selected baseline Logistic Regression as the final model
final_model = logistic_pipeline

# Generate final predictions on the test data
y_pred_final = final_model.predict(X_test)

# Evaluate the final model
print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall:", recall_score(y_test, y_pred_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

### Insight 7:

The selected final model achieved approximately **88.08%** accuracy and a macro F1-score of **0.82**. It achieved **0.86** recall for class `0` and **0.89** recall for class `1`, showing relatively balanced performance despite the class imbalance.

## Final Model Visualization

This section visualizes the final model's classification performance using a confusion matrix.

### Confusion Matrix

In [ ]:
# Visualize the confusion matrix for the final model

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_final
)

plt.title("Confusion Matrix - Final Logistic Regression Model")

plt.savefig(
    "images/final_model_result.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

### Insight 8:

The confusion matrix shows that the final Logistic Regression model correctly classified **280 of 327** class `0` observations and **1,345 of 1,518** class `1` observations. This confirms that the model maintains relatively balanced performance across both classes despite the target imbalance.

## Conclusion

The final pipeline combines numerical preprocessing, categorical encoding, spaCy lemmatization, TF-IDF feature extraction, and Logistic Regression for predicting whether a customer recommends a product.

After comparing Logistic Regression and Random Forest models, the baseline Logistic Regression was selected as the final model because it achieved the highest macro F1-score and showed more balanced performance across both classes.

The final model achieved approximately **88.08%** accuracy and a macro F1-score of **0.82**, with recall scores of **0.86** for class `0` and **0.89** for class `1`. These results show that the pipeline can effectively use both structured data and customer review text to predict product recommendations.